In [ ]:
snapshot = SimpleLogitsSnapshot(logits, x_denoising, y_denoising, id_mask)
conf_snapshot = snapshot.transform_logits(collector)
idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)
num_unmask = quota_helper.get_quota(step)
idx_transform = idx_sorted_by_conf[:, :num_unmask]

snapshot.materialize_by_idx_(idx_transform, conf_snapshot)
snapshot.update_this(1, idx_transform, y=x).update_this(1, idx_transform, p_finalized=p_finalized)

In [ ]:

import torch

len_prompt = 0
num_blocks = 0
x = y = plugin_cache_attn = None
step_refresh_remainder = 5
step_per_block = 64
model = None
collector = None
id_mask = 0
transform_only_in_h = False
sorter = None
quota_helper = None

class SimpleLogitsSnapshot:
    pass
# end

class DenoisingIDXSelector:
    pass
# end

# 正常的full一次
# 正常的拿到那个idx_transform
idx_refresh = torch.tensor([], dtype=torch.long, device=x.device)
p_finalized = torch.zeros(x.shape, dtype=torch.float64, device=x.device)

position_start, position_end = 0, len_prompt
idx_block = torch.arange(position_start, position_end, dtype=torch.long, device=x.device)
shape_target = (x.shape[0], position_end, -1)


### 还要处理snapshot扩展的问题，下午处理一下，晚上debug???

for step in range(step_per_block):

    if step == 0 or step_refresh_remainder == 0:
        idx_denoising = idx_block
        idx_current = torch.cat([idx_refresh, idx_denoising])
        logits = model(x[:, idx_current], idx_current=idx_current, shape_target=shape_target).logits
        logits_denoising = logits[:, -idx_denoising.shape[-1]:]

        logits_accumulated = torch.cat([snapshot.get_logits(), logits_denoising], dim=1)
        x_accumulated = x[:, :position_end]
        y_accumulated = y[:, :position_end]
        snapshot = SimpleLogitsSnapshot(logits_accumulated, x_accumulated, y_accumulated, id_mask)
        conf_snapshot = snapshot.transform_logits(collector)

    else:
        score_attn = plugin_cache_attn.collect_attn_from_all_blocks(model)[-1, idx_transform, :]    #[B, Q, K]
        score_attn[:,:,idx_transform] = 0 # -> let the current unmask token to be 0

        idx_denoising = DenoisingIDXSelector(score_attn) + position_start
        idx_current = torch.cat([idx_refresh, idx_denoising])

        logits = model(x[:, idx_current], idx_current=idx_current, shape_target=shape_target).logits
        logits_transform = logits[:, -idx_denoising.shape[-1]:]

        # different here compared to step == 0
        snapshot.update_logits_(idx_denoising, logits_transform)
        conf_snapshot = snapshot.transform_logits(collector)
        # different ends

        if transform_only_in_h:
            mask_nodenoising = ~torch.isin(idx_block, idx_denoising)
            conf_snapshot.scatter_(0, mask_nodenoising, conf_snapshot)
        # end
    # end

        idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)    # truth
        num_unmask = quota_helper.get_quota(step)
        idx_transform = idx_sorted_by_conf[:, :num_unmask]
        snapshot.materialize_by_idx_(idx_transform, conf_snapshot)
        snapshot.update_this(1, idx_transform, y=x).update_this(1, idx_transform, p_finalized=p_finalized)
        idx_refresh = idx_transform
    # end
# end










In [ ]:

for id_block in range(num_blocks):
    position_start = len_prompt + id_block * size_block
    position_end = position_start + size_block
    mask_mask_blk = x[:,position_start:position_end] == id_mask
    quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)


    score_attn = plugin_cache_attn.collect_attn_from_all_blocks(model)